# 02e — User Daily Rewards

Procesa `user_daily_rewards.csv` para generar features de retención y
engagement diario por usuario.

**Inputs:**
- `data/data_raw/user_daily_rewards.csv` (705K filas, 11 columnas)
- `data/data_qc/sample_user_ids.parquet`

**Outputs:**
- `data/data_qc/features_daily_rewards.parquet` (1 fila/usuario)

**Sistema de recompensas diarias:** el juego tiene un calendario de recompensas
con "sets" (ciclos de N días). El jugador reclama una recompensa cada día.
Si completa el set, empieza el siguiente. Mide retención y compromiso diario.

El campo `user_id` ya es hex limpio (no necesita mapeo de character_ids).

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'user_daily_rewards'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'user_daily_rewards.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_daily_rewards_cutoff.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True


In [3]:
# [SETUP] Carga de sample_user_ids y REFERENCE_DATE
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
[SETUP] 13:13:13 — sample_user_ids: 25,200 usuarios
[SETUP] 13:13:13 — REFERENCE_DATE: 2026-04-04 18:23:13


## 1. Carga y exploración

In [4]:
# [EXEC] Carga de user_daily_rewards.csv
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (705796, 11)
Tiempo: 1.1s
Memoria: 136.9 MB
[EXEC] 13:13:14 — Carga CSV: shape=(705796, 11), tiempo=1.1s


In [5]:
# [ANALYSIS] Tipos de datos del CSV crudo
print("TIPOS DE DATOS — CSV CRUDO")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<40} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:35]}'")

TIPOS DE DATOS — CSV CRUDO
  _id                                      dtype=str        nulls=       0 (  0.0%)  unique= 705,796  ej='ObjectId(6745f457f734be1e7138112d)'
  user_id                                  dtype=str        nulls=       0 (  0.0%)  unique= 705,795  ej='6193ea8fd66f0358fe2616ea'
  set                                      dtype=int64      nulls=       0 (  0.0%)  unique=       3  ej='1'
  current_day                              dtype=int64      nulls=       0 (  0.0%)  unique=      30  ej='3'
  last_claimed_reward_day                  dtype=int64      nulls=       0 (  0.0%)  unique=      31  ej='2'
  last_claimed_reward_time                 dtype=int64      nulls=       0 (  0.0%)  unique= 410,853  ej='1768496182'
  last_claimed_premium_reward_day          dtype=int64      nulls=       0 (  0.0%)  unique=      31  ej='1'
  last_claimed_ad_reward_day               dtype=int64      nulls=       0 (  0.0%)  unique=      31  ej='-1'
  sets_completed                   

In [6]:
# [ANALYSIS] Nulos por columna
nulls = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls['pct_nulls'] = (nulls['n_nulls'] / len(df_raw) * 100).round(2)
nulls = nulls.sort_values('pct_nulls', ascending=False)
print(nulls)
nulls.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')
log_step('ANALYSIS', 'Nulos por columna guardados')

                                 n_nulls  pct_nulls
_id                                    0        0.0
user_id                                0        0.0
set                                    0        0.0
current_day                            0        0.0
last_claimed_reward_day                0        0.0
last_claimed_reward_time               0        0.0
last_claimed_premium_reward_day        0        0.0
last_claimed_ad_reward_day             0        0.0
sets_completed                         0        0.0
updated_at                             0        0.0
created_at                             0        0.0
[ANALYSIS] 13:13:14 — Nulos por columna guardados


In [7]:
# [ANALYSIS] Distribuciones clave
print("=== set (ciclo de recompensas) ===")
print(df_raw['set'].value_counts().sort_index().head(20))
print(f"Sets distintos: {df_raw['set'].nunique()}")

print("\n=== current_day ===")
print(df_raw['current_day'].describe())

print("\n=== sets_completed ===")
print(df_raw['sets_completed'].describe())

print("\n=== last_claimed_reward_day (incluye -1 = nunca) ===")
vc_claimed = df_raw['last_claimed_reward_day'].value_counts().sort_index()
print(f"  Valor -1 (nunca): {(df_raw['last_claimed_reward_day'] == -1).sum():,}")
print(f"  Valor > 0: {(df_raw['last_claimed_reward_day'] > 0).sum():,}")
print(vc_claimed.head(10))

print("\n=== last_claimed_premium_reward_day ===")
print(f"  Valor -1 (nunca): {(df_raw['last_claimed_premium_reward_day'] == -1).sum():,}")
print(f"  Valor > 0: {(df_raw['last_claimed_premium_reward_day'] > 0).sum():,}")

print("\n=== last_claimed_ad_reward_day ===")
print(f"  Valor -1 (nunca): {(df_raw['last_claimed_ad_reward_day'] == -1).sum():,}")
print(f"  Valor > 0: {(df_raw['last_claimed_ad_reward_day'] > 0).sum():,}")

print("\n=== Registros por usuario ===")
recs_per_user = df_raw.groupby('user_id').size()
print(f"  Mediana: {recs_per_user.median():.0f}")
print(f"  Media: {recs_per_user.mean():.1f}")
print(f"  Max: {recs_per_user.max()}")
print(f"  P90: {recs_per_user.quantile(0.9):.0f}")
print(f"  P99: {recs_per_user.quantile(0.99):.0f}")

log_step('ANALYSIS', f'Distribuciones: {df_raw["set"].nunique()} sets, '
         f'recs/user mediana={recs_per_user.median():.0f}')

=== set (ciclo de recompensas) ===
set
1    702487
2      2389
3       920
Name: count, dtype: int64
Sets distintos: 3

=== current_day ===
count    705796.000000
mean          1.858730
std           2.438845
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          30.000000
Name: current_day, dtype: float64

=== sets_completed ===
count    705796.000000
mean          0.012096
std           0.234675
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          15.000000
Name: sets_completed, dtype: float64

=== last_claimed_reward_day (incluye -1 = nunca) ===
  Valor -1 (nunca): 292,182
  Valor > 0: 413,614
last_claimed_reward_day
-1    292182
 1    229154
 2     80688
 3     38829
 4     20783
 5     12288
 6      7618
 7      5103
 8      3604
 9      2465
Name: count, dtype: int64

=== last_claimed_premium_reward_day ===
  Valor -1 (nunca): 698,663
  Valor > 0: 7,133

=== last_claimed_ad_rewar

  Mediana: 1
  Media: 1.0
  Max: 2
  P90: 1
  P99: 1
[ANALYSIS] 13:13:14 — Distribuciones: 3 sets, recs/user mediana=1


## 2. Filtrado por sample_user_ids

In [8]:
# [EXEC] Filtrado por sample_user_ids
t0 = time.time()
n_before = len(df_raw)

df_filtered = df_raw[df_raw['user_id'].isin(sample_user_ids)].copy()
n_after = len(df_filtered)

n_users_with_rewards = df_filtered['user_id'].nunique()
n_users_without_rewards = N_SAMPLE - n_users_with_rewards

print(f"Filas iniciales:                 {n_before:>12,}")
print(f"Tras filtro sample_user_ids:     {n_after:>12,}  ({n_after/n_before*100:.1f}%)")
print(f"\nUsuarios CON rewards:    {n_users_with_rewards:>8,} ({n_users_with_rewards/N_SAMPLE*100:.1f}%)")
print(f"Usuarios SIN rewards:    {n_users_without_rewards:>8,} ({n_users_without_rewards/N_SAMPLE*100:.1f}%)")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Filtrado: {n_before:,} → {n_after:,} ({n_after/n_before*100:.1f}%)')

del df_raw
gc.collect()

Filas iniciales:                      705,796
Tras filtro sample_user_ids:           24,788  (3.5%)

Usuarios CON rewards:      24,788 (98.4%)
Usuarios SIN rewards:         412 (1.6%)
Tiempo: 0.1s
[EXEC] 13:13:14 — Filtrado: 705,796 → 24,788 (3.5%)


0

In [9]:
# [ANALYSIS] Estadísticas post-filtro
print(f"Filas: {len(df_filtered):,}")
print(f"Usuarios únicos: {df_filtered['user_id'].nunique():,}")
print(f"Memoria: {df_filtered.memory_usage(deep=True).sum() / 1e6:.1f} MB")

recs_per_user = df_filtered.groupby('user_id').size()
print(f"\nRegistros por usuario (post-filtro):")
print(f"  Mediana: {recs_per_user.median():.0f}")
print(f"  Media: {recs_per_user.mean():.1f}")
print(f"  Max: {recs_per_user.max()}")

print(f"\nTipos de datos:")
for col in df_filtered.columns:
    print(f"  {col:<40} {str(df_filtered[col].dtype):<15}")

log_step('ANALYSIS', f'Post-filtro: {len(df_filtered):,} filas, {df_filtered["user_id"].nunique():,} usuarios')

Filas: 24,788
Usuarios únicos: 24,788
Memoria: 5.0 MB

Registros por usuario (post-filtro):
  Mediana: 1
  Media: 1.0
  Max: 1

Tipos de datos:
  _id                                      str            
  user_id                                  str            
  set                                      int64          
  current_day                              int64          
  last_claimed_reward_day                  int64          
  last_claimed_reward_time                 int64          
  last_claimed_premium_reward_day          int64          
  last_claimed_ad_reward_day               int64          
  sets_completed                           int64          
  updated_at                               str            
  created_at                               str            
[ANALYSIS] 13:13:14 — Post-filtro: 24,788 filas, 24,788 usuarios


## 3. Agregación por usuario — ~10 features en 3 grupos

- **Grupo A:** Volumen y progresión (4 features)
- **Grupo B:** Monetización y engagement (4 features)
- **Grupo C:** Temporal (2 features)

In [10]:
# [EXEC] Grupo A: Volumen y progresión (4 features)
t0 = time.time()

group_a = df_filtered.groupby('user_id').agg(
    reward_records_total=('set', 'size'),
    reward_sets_completed_max=('sets_completed', 'max'),
    reward_current_day_max=('current_day', 'max'),
)

# Claim rate: max día reclamado / max día disponible
claimed = df_filtered[df_filtered['last_claimed_reward_day'] > 0] \
    .groupby('user_id')['last_claimed_reward_day'].max()
available = df_filtered.groupby('user_id')['current_day'].max()

group_a['reward_claim_rate'] = (claimed / available.replace(0, np.nan)).clip(0, 1).fillna(0).round(2)

print(f"Grupo A: {len(group_a):,} filas, {group_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {group_a.shape[1]} features')

Grupo A: 24,788 filas, 4 features, 0.0s
[EXEC] 13:13:14 — Grupo A (volumen): 4 features


In [11]:
# [EXEC] Grupo B: Monetización y engagement (4 features)
t0 = time.time()

has_premium = df_filtered.groupby('user_id')['last_claimed_premium_reward_day'] \
    .apply(lambda x: int((x > 0).any())).rename('reward_has_premium')

has_ad = df_filtered.groupby('user_id')['last_claimed_ad_reward_day'] \
    .apply(lambda x: int((x > 0).any())).rename('reward_has_ad')

premium_max = df_filtered[df_filtered['last_claimed_premium_reward_day'] > 0] \
    .groupby('user_id')['last_claimed_premium_reward_day'].max() \
    .rename('reward_premium_days_max')

ad_max = df_filtered[df_filtered['last_claimed_ad_reward_day'] > 0] \
    .groupby('user_id')['last_claimed_ad_reward_day'].max() \
    .rename('reward_ad_days_max')

group_b = pd.concat([has_premium, has_ad, premium_max, ad_max], axis=1)

print(f"Grupo B: {len(group_b):,} filas, {group_b.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo B (monetización): {group_b.shape[1]} features')

Grupo B: 24,788 filas, 4 features, 0.9s
[EXEC] 13:13:15 — Grupo B (monetización): 4 features


In [12]:
# [EXEC] Grupo C: Temporal (2 features)
t0 = time.time()

# Parsear created_at
df_filtered['created_dt'] = pd.to_datetime(df_filtered['created_at'], errors='coerce', utc=True).dt.tz_localize(None)

# Filtro cutoff: descartar rewards posteriores al cutoff
_cut_naive = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
n_pre_cut = len(df_filtered)
df_filtered = df_filtered[df_filtered['created_dt'].notna() & (df_filtered['created_dt'] <= _cut_naive)].copy()
log_step('EXEC', f'Filtro cutoff (created_dt <= CUTOFF): {n_pre_cut:,} → {len(df_filtered):,}')

# Último claim time como datetime
valid_times = df_filtered[df_filtered['last_claimed_reward_time'] > 0]
last_claim = valid_times.groupby('user_id')['last_claimed_reward_time'] \
    .max().apply(lambda x: pd.to_datetime(x, unit='s', errors='coerce')) \
    .rename('reward_last_claim_dt')

first_created = df_filtered.groupby('user_id')['created_dt'].min().rename('reward_first_created')

group_c = pd.concat([last_claim, first_created], axis=1)

print(f"Grupo C: {len(group_c):,} filas, {group_c.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo C (temporal): {group_c.shape[1]} features')

[EXEC] 13:13:15 — Filtro cutoff (created_dt <= CUTOFF): 24,788 → 22,941


Grupo C: 22,941 filas, 2 features, 0.5s
[EXEC] 13:13:16 — Grupo C (temporal): 2 features


In [13]:
# [EXEC] Combinar + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([group_a, group_b, group_c], axis=1)
features = features.reindex(list(sample_user_ids))

date_cols = ['reward_last_claim_dt', 'reward_first_created']
numeric_cols = [c for c in features.columns if c not in date_cols]
features[numeric_cols] = features[numeric_cols].fillna(0)

features = features.reset_index().rename(columns={'index': 'user_id'})

assert len(features) == N_SAMPLE, f"ERROR: {len(features)} != {N_SAMPLE}"

print(f"Features finales: {features.shape}")
print(f"{len(features)} filas == {N_SAMPLE} sample")
log_step('EXEC', f'Features: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (25200, 11)
25200 filas == 25200 sample
[EXEC] 13:13:16 — Features: 25,200 × 11 cols


## 4. Validación

In [14]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if dtype in ['float64', 'int64', 'int32', 'float32']:
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

TIPOS DE DATOS — FEATURES FINALES
  user_id                             dtype=str          nulls=       0
  reward_records_total                dtype=float64      nulls=       0  zeros=     412  nonzero=  24,788
  reward_sets_completed_max           dtype=float64      nulls=       0  zeros=  23,820  nonzero=   1,380
  reward_current_day_max              dtype=float64      nulls=       0  zeros=     412  nonzero=  24,788
  reward_claim_rate                   dtype=float64      nulls=       0  zeros=   1,228  nonzero=  23,972
  reward_has_premium                  dtype=float64      nulls=       0  zeros=  24,774  nonzero=     426
  reward_has_ad                       dtype=float64      nulls=       0  zeros=  20,561  nonzero=   4,639
  reward_premium_days_max             dtype=float64      nulls=       0  zeros=  24,774  nonzero=     426
  reward_ad_days_max                  dtype=float64      nulls=       0  zeros=  20,561  nonzero=   4,639
  reward_last_claim_dt                dtype=da

In [15]:
# [ANALYSIS] Nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos (esperado solo en fechas):")
    print(nulls_f)
else:
    print("No hay nulos")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos: {len(nulls_f)} columnas con nulos')

Columnas con nulos (esperado solo en fechas):
                      n_nulls  pct_nulls
reward_last_claim_dt     3000      11.90
reward_first_created     2259       8.96
[ANALYSIS] 13:13:16 — Nulos: 2 columnas con nulos


In [16]:
# [ANALYSIS] Estadísticas descriptivas e histogramas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')

key_features = [
    'reward_records_total', 'reward_sets_completed_max', 'reward_current_day_max',
    'reward_claim_rate', 'reward_has_premium', 'reward_has_ad',
    'reward_premium_days_max', 'reward_ad_days_max',
]

n_plots = len(key_features)
ncols = 3
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, key_features):
    if feat in features.columns:
        data = features[feat][features[feat] > 0]
        if len(data) > 0:
            ax.hist(np.log1p(data), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(data):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

for ax in axes_flat[len(key_features):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Estadísticas y histogramas guardados')

                             count  mean   std  min  25%  50%  75%   90%   99%   max
reward_records_total       25200.0  0.98  0.13  0.0  1.0  1.0  1.0   1.0   1.0   1.0
reward_sets_completed_max  25200.0  0.16  0.98  0.0  0.0  0.0  0.0   0.0   5.0  15.0
reward_current_day_max     25200.0  5.04  5.39  0.0  2.0  3.0  6.0  12.0  27.0  30.0
reward_claim_rate          25200.0  0.91  0.23  0.0  1.0  1.0  1.0   1.0   1.0   1.0
reward_has_premium         25200.0  0.02  0.13  0.0  0.0  0.0  0.0   0.0   1.0   1.0
reward_has_ad              25200.0  0.18  0.39  0.0  0.0  0.0  0.0   1.0   1.0   1.0
reward_premium_days_max    25200.0  0.13  1.37  0.0  0.0  0.0  0.0   0.0   5.0  29.0
reward_ad_days_max         25200.0  1.22  3.91  0.0  0.0  0.0  0.0   3.0  22.0  30.0


[ANALYSIS] 13:13:16 — Estadísticas y histogramas guardados


In [17]:
# [EXEC] Guardar features_daily_rewards_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
log_step('EXEC', f'features_daily_rewards_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")

[EXEC] 13:13:16 — features_daily_rewards_cutoff.parquet: (25200, 11), 1.1 MB
Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_daily_rewards_cutoff.parquet (1.1 MB)


In [18]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df_filtered
gc.collect()
print("Memoria liberada")

[EXEC] 13:13:16 — sample_head.csv guardado
Memoria liberada


## 5. Informe de ejecución

In [19]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con = features[features['reward_records_total'] > 0].shape[0]
n_sin = features[features['reward_records_total'] == 0].shape[0]

lines = []
lines += [
    "# Informe de ejecucion — user_daily_rewards.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02e_user_daily_rewards.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/user_daily_rewards.csv (89 MB, {n_before:,} filas)",
    f"**Parquet de salida:** data/data_qc/features_daily_rewards_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Sistema de recompensas diarias",
    "",
    "El juego tiene un calendario de recompensas con 'sets' (ciclos de N días).",
    "El jugador reclama una recompensa cada día. Si completa el set, empieza",
    "el siguiente. Esto mide retención y compromiso diario.",
    "",
    "- `set`: número del ciclo de recompensas",
    "- `current_day`: día actual dentro del set",
    "- `last_claimed_reward_day`: último día reclamado (-1 = nunca)",
    "- `last_claimed_premium_reward_day`: último día premium reclamado (-1 = nunca)",
    "- `last_claimed_ad_reward_day`: último día de reward por anuncio (-1 = nunca)",
    "- `sets_completed`: total de sets completados (acumulativo)",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before:,} registros de recompensas diarias. Tras filtrar",
    f"por los {N_SAMPLE:,} usuarios del sample, quedaron {n_after:,} registros.",
    f"Se generaron {features.shape[1] - 1} features en 3 grupos.",
    "",
    f"- Usuarios con rewards: {n_con:,} ({n_con/len(features)*100:.1f}%)",
    f"- Usuarios sin rewards: {n_sin:,} ({n_sin/len(features)*100:.1f}%)",
    "",
    "---", "",
    "## Constantes",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    "",
    "---", "",
    "## Pasos ejecutados",
    "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado",
    "",
    "| Paso | Filas |",
    "|---|---|",
    f"| CSV original | {n_before:,} |",
    f"| Filtro sample_user_ids | {n_after:,} |",
    "",
    "---", "",
    f"## Features generadas ({features.shape[1] - 1} + user_id)",
    "",
    "### Grupo A — Volumen y progresión (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `reward_records_total` | Registros de recompensa |",
    "| `reward_sets_completed_max` | Sets completados (max) |",
    "| `reward_current_day_max` | Día más alto alcanzado |",
    "| `reward_claim_rate` | % días reclamados vs disponibles [0-1] |",
    "",
    "### Grupo B — Monetización (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `reward_has_premium` | 1 si reclamó reward premium |",
    "| `reward_has_ad` | 1 si vio anuncios por rewards |",
    "| `reward_premium_days_max` | Días premium reclamados (max) |",
    "| `reward_ad_days_max` | Días con anuncio reclamado (max) |",
    "",
    "### Grupo C — Temporal (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `reward_last_claim_dt` | Fecha último claim |",
    "| `reward_first_created` | Fecha primer registro |",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- user_id ya en formato hex limpio (sin mapeo necesario)",
    "- Sin nulos en el CSV original",
    "- reward_claim_rate captura consistencia de login",
    "- reward_has_premium y reward_has_ad como señales de monetización",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- Valor -1 en last_claimed_*_day tratado como 'nunca' (no contado como día)",
    "- reward_claim_rate clipeado a [0, 1]",
    "- sets_completed: se toma el max por usuario (es acumulativo)",
    "- Usuarios sin rewards mantienen fila con valores 0",
    "",
    "---", "",
]

feat_size = PARQUET_FEATURES.stat().st_size / 1e6
lines += [
    "## Archivos generados",
    "",
    "| Archivo | Tamaño |",
    "|---|---|",
    f"| features_daily_rewards_cutoff.parquet ({features.shape[1]} cols) | {feat_size:.1f} MB |",
    "| nulos_por_columna_raw.csv | - |",
    "| nulos_features_final.csv | - |",
    "| features_describe.csv | - |",
    "| features_distributions.png | - |",
    "| sample_head.csv | - |",
    "| execution_report.md | - |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    f"- ~{n_sin/len(features)*100:.0f}% usuarios sin rewards (features a cero)",
    "- `reward_claim_rate` es proxy de consistencia de login diario",
    "- `reward_has_premium = 1` indica usuario con rewards premium",
    "- `reward_has_ad = 1` indica usuario que ve anuncios voluntariamente",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] Investigar correlación reward_claim_rate vs churn",
    "- [ ] Considerar feature de 'racha' (días consecutivos reclamados)",
    "- [ ] Evaluar si reward_has_ad es señal de F2P comprometido (ve ads para avanzar)",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"user_daily_rewards.csv ({n_before:,} filas)",
    "    │",
    f"    ├─ Filtrar por sample_user_ids  (-{n_before - n_after:,})",
    "    ▼",
    f"Filtrado ({n_after:,} filas)",
    "    │",
    "    ├─ Grupo A: volumen (4)",
    "    ├─ Grupo B: monetización (4)",
    "    ├─ Grupo C: temporal (2)",
    "    ├─ Reindex con sample",
    "    ▼",
    f"features_daily_rewards_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero",
    "2. Ejecutar 02e_user_daily_rewards.ipynb",
    f"3. Verificar: features_daily_rewards_cutoff.parquet == {N_SAMPLE:,} filas",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_daily_rewards/execution_report.md
[REPORT] 13:13:16 — execution_report.md generado


In [20]:
# [REPORT] Resumen final
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02e_user_daily_rewards")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original    : {n_before:,}")
print(f"  Filas filtradas       : {n_after:,}")
print(f"  Usuarios con rewards  : {n_users_with_rewards:,} ({n_users_with_rewards/N_SAMPLE*100:.1f}%)")
print(f"  Usuarios sin rewards  : {n_users_without_rewards:,} ({n_users_without_rewards/N_SAMPLE*100:.1f}%)")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print()
print("Archivos generados:")
print(f"  features_daily_rewards_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02e_user_daily_rewards
  Tiempo total          : 0m 3s
  Filas CSV original    : 705,796
  Filas filtradas       : 24,788
  Usuarios con rewards  : 24,788 (98.4%)
  Usuarios sin rewards  : 412 (1.6%)
  Filas features final  : 25,200 (== 25,200 sample)
  Columnas features     : 11

Archivos generados:
  features_daily_rewards_cutoff.parquet (1.1 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_daily_rewards
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [21]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02e_user_daily_rewards.ipynb'
html_path = REPORT_DIR / '02e_user_daily_rewards_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(705796, 11),
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02e_user_daily_rewards_run.html (0.4 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_daily_rewards/execution_report.md
